In [18]:
import pandas as pd

df = pd.read_csv('/healthcare-dataset-stroke-data.csv')

if 'bmi' in df.columns:
  df['bmi'] = df['bmi'].fillna(df['bmi'].median())

if 'id' in df.columns:
  df = df.drop(columns=['id'])

df_encoded = pd.get_dummies(df, drop_first=True)

print(
    f'¡Carga y preprocesamiento exitosos! Dimensiones de df_encoded:'
    f' {df_encoded.shape}'
)
df_encoded.head()

¡Carga y preprocesamiento exitosos! Dimensiones de df_encoded: (5110, 17)


,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke,gender_Male,gender_Other,ever_married_Yes,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Urban,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
0,67.0,0,1,228.69,36.6,1,True,False,True,False,True,False,False,True,True,False,False
1,61.0,0,0,202.21,28.1,1,False,False,True,False,False,True,False,False,False,True,False
2,80.0,0,1,105.92,32.5,1,True,False,True,False,True,False,False,False,False,True,False
3,49.0,0,0,171.23,34.4,1,False,False,True,False,True,False,False,True,False,False,True
4,79.0,1,0,174.12,24.0,1,False,False,True,False,False,True,False,False,False,True,False


# División de datos y Escalado (Preparación para Deep Learning)




In [15]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df_encoded.drop(columns=['stroke'])
y = df_encoded['stroke']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Dimensiones de X_train: {X_train.shape}")
print(f"Dimensiones de X_test: {X_test.shape}")
print("\nDistribución en y_train:\n", y_train.value_counts(normalize=True))
print("\nDistribución en y_test:\n", y_test.value_counts(normalize=True))


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Dimensiones de X_train: (4088, 16)
Dimensiones de X_test: (1022, 16)

Distribución en y_train:
 stroke
0    0.951321
1    0.048679
Name: proportion, dtype: float64

Distribución en y_test:
 stroke
0    0.951076
1    0.048924
Name: proportion, dtype: float64


# Diseño de la Arquitectura y Selección de Funciones (Keras / TensorFlow)

In [16]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([

    layers.Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    layers.Dropout(0.3),


    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),


    layers.Dense(1, activation='sigmoid')
])

model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,201 (12.50 KB)

 Trainable params: 3,201 (12.50 KB)

 Non-trainable params: 0 (0.00 B)

# Entrenamiento, Métricas y Optimizadores


In [17]:
from tensorflow.keras.callbacks import EarlyStopping

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name='accuracy'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc')
    ]
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/100
103/103 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.8676 - auc: 0.5113 - loss: 0.3536 - precision: 0.0568 - recall: 0.1184 - val_accuracy: 0.9425 - val_auc: 0.6352 - val_loss: 0.2143 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/100
103/103 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9532 - auc: 0.6683 - loss: 0.1879 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_accuracy: 0.9425 - val_auc: 0.8002 - val_loss: 0.1918 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 3/100
103/103 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9535 - auc: 0.7316 - loss: 0.1776 - precision: 0.5000 - recall: 0.0066 - val_accuracy: 0.9425 - val_auc: 0.8277 - val_loss: 0.1833 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 4/100
103/103 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9535 - auc: 0.7702 - loss: 0.1701 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_accuracy: 0.9425 - val_auc: 0.8378 - val_loss: 0.1788 - val_precision: 0.0000e+00 - val